# 🚦 TransitLK — AI-Based Traffic Signal Control
## Phase 1 · Approach A: Queue Length Based DQN

This notebook trains a **Deep Q-Network (DQN)** agent to control a traffic signal
for a simulated Sri Lankan road with **two lanes** in one direction.

**Key features:**
- Lightweight custom traffic simulation (no external simulator needed)
- PyTorch DQN with experience replay, target network, ε-greedy exploration
- 4 performance metrics tracked: Avg Queue Length, Avg Wait Time, Throughput, Efficiency
- Manual override capability
- GPU-accelerated training when available

---
**How to run on Kaggle:**
1. Toggle the **GPU accelerator** under *Settings → Accelerator → GPU*
2. Click **Run All** (▶▶)
3. Training results and plots appear at the bottom


## 1 · Install Dependencies

In [ ]:
# PyTorch is pre-installed on Kaggle GPU kernels.
# Just verify the setup:
import torch
print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU device      : {torch.cuda.get_device_name(0)}")


## 2 · Environment — Traffic Simulation

In [ ]:
"""
Traffic Signal Environment for DQN-based control.

Simulates a Sri Lankan main road with two lanes in one direction.
Vehicles arrive randomly (Poisson process) and queue up when the
signal is red. The agent controls which lane gets the green signal
and for how long.
"""

import numpy as np
from dataclasses import dataclass, field
from typing import Tuple, Dict, List, Optional


@dataclass
class Vehicle:
    """Represents a single vehicle in the simulation."""
    arrival_time: int
    departure_time: int = -1


@dataclass
class LaneConfig:
    """Configuration for a single lane."""
    arrival_rate: float = 3.0
    saturation_rate: int = 2


@dataclass
class EnvConfig:
    """Full environment configuration."""
    num_lanes: int = 2
    episode_length: int = 500
    min_green_duration: int = 5
    short_green: int = 10
    long_green: int = 20
    yellow_phase: int = 2
    lane_configs: List[LaneConfig] = field(default_factory=lambda: [
        LaneConfig(arrival_rate=3.0, saturation_rate=2),
        LaneConfig(arrival_rate=2.0, saturation_rate=2),
    ])


class TrafficEnv:
    """
    Gym-style traffic signal environment.

    State:  [queue_lane_0, queue_lane_1] — normalised queue lengths
    Actions:
        0 — Green for Lane 0, short duration
        1 — Green for Lane 0, long duration
        2 — Green for Lane 1, short duration
        3 — Green for Lane 1, long duration
    Reward: −(total waiting vehicles) each step
    """

    def __init__(self, config: Optional[EnvConfig] = None):
        self.config = config or EnvConfig()
        self.num_actions = self.config.num_lanes * 2
        self._queues: List[List[Vehicle]] = []
        self._current_step: int = 0
        self._green_lane: int = 0
        self._green_remaining: int = 0
        self._yellow_remaining: int = 0
        self._phase_locked: bool = False
        self.override_active: bool = False
        self._override_lane: int = 0
        self._override_remaining: int = 0
        self._total_arrived: int = 0
        self._total_departed: int = 0
        self._step_queue_log: List[List[int]] = []
        self._step_departed_log: List[int] = []
        self._step_arrived_log: List[int] = []
        self._vehicle_wait_times: List[int] = []
        self._rng = np.random.default_rng()

    def seed(self, seed: int):
        self._rng = np.random.default_rng(seed)

    def reset(self) -> np.ndarray:
        self._queues = [[] for _ in range(self.config.num_lanes)]
        self._current_step = 0
        self._green_lane = 0
        self._green_remaining = self.config.short_green
        self._yellow_remaining = 0
        self._phase_locked = True
        self.override_active = False
        self._override_lane = 0
        self._override_remaining = 0
        self._total_arrived = 0
        self._total_departed = 0
        self._step_queue_log = []
        self._step_departed_log = []
        self._step_arrived_log = []
        self._vehicle_wait_times = []
        return self._get_state()

    def step(self, action: int) -> Tuple[np.ndarray, float, bool, Dict]:
        step_arrived = self._process_arrivals()
        self._process_signal(action)
        step_departed = self._process_departures()
        queue_snapshot = self.get_queue_lengths()
        self._step_queue_log.append(queue_snapshot.copy())
        self._step_departed_log.append(step_departed)
        self._step_arrived_log.append(step_arrived)
        total_waiting = sum(queue_snapshot)
        reward = -float(total_waiting)
        self._current_step += 1
        done = self._current_step >= self.config.episode_length
        info = {
            "step": self._current_step,
            "queue_lengths": queue_snapshot,
            "departed_this_step": step_departed,
            "arrived_this_step": step_arrived,
            "green_lane": self._green_lane,
            "green_remaining": self._green_remaining,
            "override_active": self.override_active,
            "total_arrived": self._total_arrived,
            "total_departed": self._total_departed,
        }
        return self._get_state(), reward, done, info

    def _process_arrivals(self) -> int:
        total = 0
        for i, lane_cfg in enumerate(self.config.lane_configs):
            n = self._rng.poisson(lane_cfg.arrival_rate)
            for _ in range(n):
                self._queues[i].append(Vehicle(arrival_time=self._current_step))
            total += n
        self._total_arrived += total
        return total

    def _process_signal(self, action: int):
        if self.override_active:
            self._green_lane = self._override_lane
            self._override_remaining -= 1
            if self._override_remaining <= 0:
                self.override_active = False
                self._yellow_remaining = self.config.yellow_phase
                self._phase_locked = True
            return
        if self._yellow_remaining > 0:
            self._yellow_remaining -= 1
            self._phase_locked = self._yellow_remaining > 0
            return
        if self._green_remaining > 0:
            self._green_remaining -= 1
            if self._green_remaining > 0:
                return
        lane, duration = self._decode_action(action)
        if lane != self._green_lane:
            self._yellow_remaining = self.config.yellow_phase
            self._phase_locked = True
            self._green_lane = lane
        self._green_remaining = duration

    def _process_departures(self) -> int:
        if self._yellow_remaining > 0 or self.override_active:
            if self.override_active:
                return self._depart_from_lane(self._override_lane)
            return 0
        return self._depart_from_lane(self._green_lane)

    def _depart_from_lane(self, lane: int) -> int:
        sat = self.config.lane_configs[lane].saturation_rate
        departed = 0
        while self._queues[lane] and departed < sat:
            vehicle = self._queues[lane].pop(0)
            vehicle.departure_time = self._current_step
            wait = vehicle.departure_time - vehicle.arrival_time
            self._vehicle_wait_times.append(wait)
            self._total_departed += 1
            departed += 1
        return departed

    def _decode_action(self, action: int) -> Tuple[int, int]:
        lane = action // 2
        duration = (
            self.config.short_green if action % 2 == 0
            else self.config.long_green
        )
        return lane, duration

    def _get_state(self) -> np.ndarray:
        max_q = 50.0
        queues = [len(q) / max_q for q in self._queues]
        return np.array(queues, dtype=np.float32)

    def get_queue_lengths(self) -> List[int]:
        return [len(q) for q in self._queues]

    def get_episode_stats(self) -> Dict:
        avg_queue = (
            np.mean([sum(q) for q in self._step_queue_log])
            if self._step_queue_log else 0.0
        )
        avg_wait = (
            np.mean(self._vehicle_wait_times)
            if self._vehicle_wait_times else 0.0
        )
        throughput = self._total_departed
        efficiency = (
            self._total_departed / self._total_arrived
            if self._total_arrived > 0 else 0.0
        )
        return {
            "avg_queue_length": float(avg_queue),
            "avg_waiting_time": float(avg_wait),
            "throughput": int(throughput),
            "efficiency": float(efficiency),
            "total_arrived": int(self._total_arrived),
            "total_departed": int(self._total_departed),
        }

    def set_signal(self, lane: int, duration: int):
        assert 0 <= lane < self.config.num_lanes, f"Invalid lane {lane}"
        assert duration > 0, "Duration must be positive"
        self.override_active = True
        self._override_lane = lane
        self._override_remaining = duration
        self._green_lane = lane
        self._green_remaining = 0
        self._yellow_remaining = 0
        self._phase_locked = True

print("✅ TrafficEnv ready.")

## 3 · DQN Agent

In [ ]:
"""
Deep Q-Network (DQN) Agent for traffic signal control.
"""

import random
from collections import deque

import torch
import torch.nn as nn
import torch.optim as optim


class QNetwork(nn.Module):
    def __init__(self, state_dim: int, action_dim: int, hidden: int = 128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, action_dim),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class ReplayBuffer:
    def __init__(self, capacity: int = 10_000):
        self.buffer = deque(maxlen=capacity)

    def push(self, state, action, reward, next_state, done):
        self.buffer.append((state, action, reward, next_state, done))

    def sample(self, batch_size: int):
        batch = random.sample(self.buffer, batch_size)
        states, actions, rewards, next_states, dones = zip(*batch)
        return (
            np.array(states, dtype=np.float32),
            np.array(actions, dtype=np.int64),
            np.array(rewards, dtype=np.float32),
            np.array(next_states, dtype=np.float32),
            np.array(dones, dtype=np.float32),
        )

    def __len__(self):
        return len(self.buffer)


class DQNAgent:
    def __init__(
        self,
        state_dim: int = 2,
        action_dim: int = 4,
        gamma: float = 0.99,
        lr: float = 0.001,
        batch_size: int = 64,
        buffer_capacity: int = 10_000,
        buffer_warmup: int = 500,
        epsilon_start: float = 1.0,
        epsilon_end: float = 0.01,
        epsilon_decay: float = 0.995,
        target_update: int = 10,
        tau: float = 1.0,
        device: Optional[str] = None,
    ):
        if device is None:
            self.device = torch.device(
                "cuda" if torch.cuda.is_available() else "cpu"
            )
        else:
            self.device = torch.device(device)

        self.state_dim = state_dim
        self.action_dim = action_dim
        self.gamma = gamma
        self.lr = lr
        self.batch_size = batch_size
        self.buffer_warmup = buffer_warmup
        self.epsilon = epsilon_start
        self.epsilon_end = epsilon_end
        self.epsilon_decay = epsilon_decay
        self.target_update = target_update
        self.tau = tau

        self.q_net = QNetwork(state_dim, action_dim).to(self.device)
        self.target_net = QNetwork(state_dim, action_dim).to(self.device)
        self.target_net.load_state_dict(self.q_net.state_dict())
        self.target_net.eval()

        self.optimizer = optim.Adam(self.q_net.parameters(), lr=self.lr)
        self.loss_fn = nn.SmoothL1Loss()

        self.buffer = ReplayBuffer(buffer_capacity)
        self.train_step_count = 0

    def select_action(self, state: np.ndarray, epsilon: Optional[float] = None) -> int:
        eps = epsilon if epsilon is not None else self.epsilon
        if random.random() < eps:
            return random.randrange(self.action_dim)
        with torch.no_grad():
            state_t = torch.FloatTensor(state).unsqueeze(0).to(self.device)
            q_values = self.q_net(state_t)
            return int(q_values.argmax(dim=1).item())

    def store_transition(self, state, action, reward, next_state, done):
        self.buffer.push(state, action, reward, next_state, done)

    def train_step(self) -> Optional[float]:
        if len(self.buffer) < self.buffer_warmup:
            return None
        states, actions, rewards, next_states, dones = self.buffer.sample(
            self.batch_size
        )
        states_t = torch.FloatTensor(states).to(self.device)
        actions_t = torch.LongTensor(actions).to(self.device)
        rewards_t = torch.FloatTensor(rewards).to(self.device)
        next_states_t = torch.FloatTensor(next_states).to(self.device)
        dones_t = torch.FloatTensor(dones).to(self.device)

        q_values = self.q_net(states_t)
        q_taken = q_values.gather(1, actions_t.unsqueeze(1))

        with torch.no_grad():
            next_q = self.target_net(next_states_t)
            next_q_max = next_q.max(dim=1)[0]
            target = rewards_t + self.gamma * next_q_max * (1.0 - dones_t)

        loss = self.loss_fn(q_taken.squeeze(), target)

        self.optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(self.q_net.parameters(), max_norm=1.0)
        self.optimizer.step()

        self.train_step_count += 1
        return loss.item()

    def update_target_network(self):
        if self.tau >= 1.0:
            self.target_net.load_state_dict(self.q_net.state_dict())
        else:
            for tp, sp in zip(
                self.target_net.parameters(), self.q_net.parameters()
            ):
                tp.data.copy_(self.tau * sp.data + (1.0 - self.tau) * tp.data)

    def decay_epsilon(self):
        self.epsilon = max(self.epsilon_end, self.epsilon * self.epsilon_decay)

    def save(self, path: str):
        torch.save({
            "q_net_state_dict": self.q_net.state_dict(),
            "target_net_state_dict": self.target_net.state_dict(),
            "optimizer_state_dict": self.optimizer.state_dict(),
            "epsilon": self.epsilon,
            "train_step_count": self.train_step_count,
        }, path)
        print(f"[DQN] Model saved to {path}")

    def load(self, path: str):
        checkpoint = torch.load(path, map_location=self.device, weights_only=True)
        self.q_net.load_state_dict(checkpoint["q_net_state_dict"])
        self.target_net.load_state_dict(checkpoint["target_net_state_dict"])
        self.optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
        self.epsilon = checkpoint.get("epsilon", self.epsilon_end)
        self.train_step_count = checkpoint.get("train_step_count", 0)
        print(f"[DQN] Model loaded from {path}")

print(f"✅ DQN Agent ready.  Device: {torch.device('cuda' if torch.cuda.is_available() else 'cpu')}")

## 4 · Manual Override

In [ ]:
def override_signal(env, lane: int, green_duration: int) -> dict:
    """Force a specific lane to green for a given duration, bypassing the DQN."""
    if lane < 0 or lane >= env.config.num_lanes:
        raise ValueError(f"Invalid lane {lane}. Must be 0 to {env.config.num_lanes - 1}.")
    if green_duration <= 0:
        raise ValueError("Green duration must be a positive integer.")
    env.set_signal(lane, green_duration)
    return {
        "status": "override_active",
        "lane": lane,
        "duration": green_duration,
        "message": f"Manual override activated: Lane {lane} → GREEN for {green_duration} steps.",
    }


def cancel_override(env) -> dict:
    """Cancel an active manual override, returning control to the DQN agent."""
    was_active = env.override_active
    env.override_active = False
    env._override_remaining = 0
    env._phase_locked = False
    return {
        "status": "override_cancelled" if was_active else "no_override_active",
        "message": (
            "Manual override cancelled. AI agent will resume control."
            if was_active else "No override was active."
        ),
    }

print("✅ Manual Override functions ready.")

## 5 · Metrics Tracker

In [ ]:
import json
import matplotlib
import matplotlib.pyplot as plt
%matplotlib inline


class MetricsTracker:
    def __init__(self):
        self._step_queues = []
        self._step_departed = []
        self._step_arrived = []
        self._wait_times = []
        self.history = []
        self.episode_losses = []
        self._current_losses = []

    def record_step(self, queue_lengths, vehicles_departed, vehicles_arrived, wait_times=None):
        self._step_queues.append(sum(queue_lengths))
        self._step_departed.append(vehicles_departed)
        self._step_arrived.append(vehicles_arrived)
        if wait_times:
            self._wait_times.extend(wait_times)

    def record_loss(self, loss):
        self._current_losses.append(loss)

    def end_episode(self, env_stats=None):
        if env_stats:
            metrics = {
                "episode": len(self.history) + 1,
                "avg_queue_length": env_stats["avg_queue_length"],
                "avg_waiting_time": env_stats["avg_waiting_time"],
                "throughput": env_stats["throughput"],
                "efficiency": env_stats["efficiency"],
                "total_arrived": env_stats["total_arrived"],
                "total_departed": env_stats["total_departed"],
            }
        else:
            total_arrived = sum(self._step_arrived)
            total_departed = sum(self._step_departed)
            metrics = {
                "episode": len(self.history) + 1,
                "avg_queue_length": float(np.mean(self._step_queues)) if self._step_queues else 0.0,
                "avg_waiting_time": float(np.mean(self._wait_times)) if self._wait_times else 0.0,
                "throughput": total_departed,
                "efficiency": total_departed / total_arrived if total_arrived > 0 else 0.0,
                "total_arrived": total_arrived,
                "total_departed": total_departed,
            }
        if self._current_losses:
            metrics["avg_loss"] = float(np.mean(self._current_losses))
            self.episode_losses.append(metrics["avg_loss"])
        else:
            metrics["avg_loss"] = None
        self.history.append(metrics)
        self._step_queues = []
        self._step_departed = []
        self._step_arrived = []
        self._wait_times = []
        self._current_losses = []
        return metrics

    def get_history(self):
        return self.history

    @staticmethod
    def _smooth(values, window=20):
        smoothed = []
        for i in range(len(values)):
            start = max(0, i - window + 1)
            smoothed.append(np.mean(values[start : i + 1]))
        return smoothed

    def plot_metrics(self):
        if not self.history:
            print("No data to plot."); return
        episodes = [m["episode"] for m in self.history]
        avg_q = [m["avg_queue_length"] for m in self.history]
        avg_w = [m["avg_waiting_time"] for m in self.history]
        throughput = [m["throughput"] for m in self.history]
        efficiency = [m["efficiency"] for m in self.history]

        fig, axes = plt.subplots(2, 2, figsize=(14, 10))
        fig.suptitle("DQN Traffic Signal Control — Learning Curves", fontsize=16, fontweight="bold")

        for ax, data, color, title, ylabel in [
            (axes[0,0], avg_q, "#e74c3c", "Average Queue Length", "Avg Queue Length"),
            (axes[0,1], avg_w, "#3498db", "Average Waiting Time", "Avg Wait (steps)"),
            (axes[1,0], throughput, "#2ecc71", "Throughput (Vehicles Passed)", "Vehicles"),
            (axes[1,1], efficiency, "#9b59b6", "Efficiency (Passed / Arrived)", "Ratio"),
        ]:
            ax.plot(episodes, data, color=color, linewidth=1.2, alpha=0.7)
            if len(data) >= 10:
                ax.plot(episodes, self._smooth(data), color=color, linewidth=2.5, label="Smoothed")
            ax.set_title(title); ax.set_xlabel("Episode"); ax.set_ylabel(ylabel)
            ax.grid(True, alpha=0.3); ax.legend()

        axes[1,1].axhline(y=1.0, color="gray", linestyle="--", alpha=0.5, label="Ideal")
        axes[1,1].set_ylim(0, 1.1)
        plt.tight_layout(rect=[0, 0, 1, 0.95])
        plt.show()

    def plot_loss(self):
        if not self.episode_losses:
            print("No loss data to plot."); return
        fig, ax = plt.subplots(figsize=(10, 5))
        eps = list(range(1, len(self.episode_losses) + 1))
        ax.plot(eps, self.episode_losses, color="#e67e22", linewidth=1.2, alpha=0.7)
        if len(self.episode_losses) >= 10:
            ax.plot(eps, self._smooth(self.episode_losses), color="#e67e22", linewidth=2.5, label="Smoothed")
        ax.set_title("Training Loss", fontsize=14, fontweight="bold")
        ax.set_xlabel("Episode"); ax.set_ylabel("Avg Loss")
        ax.grid(True, alpha=0.3); ax.legend()
        plt.tight_layout(); plt.show()

print("✅ MetricsTracker ready.")

## 6 · Configuration

In [ ]:
# =====================================================
#  Hyperparameters — tune these as needed
# =====================================================

NUM_EPISODES      = 500       # training episodes
EPISODE_LENGTH    = 500       # simulation steps per episode
SHORT_GREEN       = 10        # short green duration (steps)
LONG_GREEN        = 20        # long green duration (steps)
YELLOW_PHASE      = 2         # amber transition steps
LANE_0_ARRIVAL    = 3.0       # Poisson λ for Lane 0 (heavier traffic)
LANE_1_ARRIVAL    = 2.0       # Poisson λ for Lane 1 (lighter traffic)
SATURATION_RATE   = 2         # max vehicles departing per green step

# DQN hyperparameters
GAMMA             = 0.99
LEARNING_RATE     = 0.001
BATCH_SIZE        = 64
BUFFER_CAPACITY   = 10_000
BUFFER_WARMUP     = 500
EPSILON_START     = 1.0
EPSILON_END       = 0.01
EPSILON_DECAY     = 0.995
TARGET_UPDATE     = 10        # episodes between target net sync

print("✅ Configuration set.")
print(f"   Training for {NUM_EPISODES} episodes × {EPISODE_LENGTH} steps each")

## 7 · Training Loop

In [ ]:
import time

# Build environment
env_config = EnvConfig(
    episode_length=EPISODE_LENGTH,
    short_green=SHORT_GREEN,
    long_green=LONG_GREEN,
    yellow_phase=YELLOW_PHASE,
    lane_configs=[
        LaneConfig(arrival_rate=LANE_0_ARRIVAL, saturation_rate=SATURATION_RATE),
        LaneConfig(arrival_rate=LANE_1_ARRIVAL, saturation_rate=SATURATION_RATE),
    ],
)
env = TrafficEnv(env_config)

# Build agent
agent = DQNAgent(
    state_dim=env_config.num_lanes,
    action_dim=env_config.num_lanes * 2,
    gamma=GAMMA,
    lr=LEARNING_RATE,
    batch_size=BATCH_SIZE,
    buffer_capacity=BUFFER_CAPACITY,
    buffer_warmup=BUFFER_WARMUP,
    epsilon_start=EPSILON_START,
    epsilon_end=EPSILON_END,
    epsilon_decay=EPSILON_DECAY,
    target_update=TARGET_UPDATE,
)

tracker = MetricsTracker()

print("=" * 65)
print("  TransitLK — DQN Traffic Signal Control  (Training)")
print("=" * 65)
print(f"  Device  : {agent.device}")
print(f"  Episodes: {NUM_EPISODES}")
print(f"  Steps/episode: {EPISODE_LENGTH}")
print("=" * 65)
print()

best_efficiency = 0.0
start_time = time.time()

for ep in range(1, NUM_EPISODES + 1):
    state = env.reset()
    episode_reward = 0.0
    done = False

    while not done:
        action = agent.select_action(state)
        next_state, reward, done, info = env.step(action)
        agent.store_transition(state, action, reward, next_state, done)
        loss = agent.train_step()
        if loss is not None:
            tracker.record_loss(loss)
        tracker.record_step(
            queue_lengths=info["queue_lengths"],
            vehicles_departed=info["departed_this_step"],
            vehicles_arrived=info["arrived_this_step"],
        )
        episode_reward += reward
        state = next_state

    env_stats = env.get_episode_stats()
    ep_metrics = tracker.end_episode(env_stats)
    agent.decay_epsilon()

    if ep % agent.target_update == 0:
        agent.update_target_network()

    if ep_metrics["efficiency"] > best_efficiency:
        best_efficiency = ep_metrics["efficiency"]
        agent.save("best_checkpoint.pth")

    if ep % 25 == 0 or ep == 1:
        elapsed = time.time() - start_time
        print(
            f"Ep {ep:4d}/{NUM_EPISODES} | "
            f"Reward: {episode_reward:8.1f} | "
            f"AvgQ: {ep_metrics['avg_queue_length']:6.2f} | "
            f"AvgWait: {ep_metrics['avg_waiting_time']:6.2f} | "
            f"Throughput: {ep_metrics['throughput']:5d} | "
            f"Efficiency: {ep_metrics['efficiency']:.3f} | "
            f"ε: {agent.epsilon:.3f} | "
            f"{elapsed:.1f}s"
        )

agent.save("final_model.pth")
total_time = time.time() - start_time
print(f"\n✅ Training complete in {total_time:.1f}s | Best Efficiency: {best_efficiency:.4f}")

## 8 · Learning Curves

In [ ]:
tracker.plot_metrics()

In [ ]:
tracker.plot_loss()

## 9 · Evaluation (Greedy Policy)

In [ ]:
# Evaluate the best model with ε = 0 (fully greedy)
agent.load("best_checkpoint.pth")

eval_tracker = MetricsTracker()
EVAL_EPISODES = 20

print("=" * 65)
print("  Evaluation — Greedy Policy (ε = 0)")
print("=" * 65)

for ep in range(1, EVAL_EPISODES + 1):
    state = env.reset()
    done = False
    while not done:
        action = agent.select_action(state, epsilon=0.0)
        next_state, reward, done, info = env.step(action)
        eval_tracker.record_step(
            queue_lengths=info["queue_lengths"],
            vehicles_departed=info["departed_this_step"],
            vehicles_arrived=info["arrived_this_step"],
        )
        state = next_state
    env_stats = env.get_episode_stats()
    m = eval_tracker.end_episode(env_stats)
    print(
        f"  Ep {ep:2d} | AvgQ: {m['avg_queue_length']:6.2f} | "
        f"AvgWait: {m['avg_waiting_time']:6.2f} | "
        f"Throughput: {m['throughput']:5d} | "
        f"Efficiency: {m['efficiency']:.3f}"
    )

# Summary
history = eval_tracker.get_history()
print()
print("-" * 65)
print("  Evaluation Summary")
print("-" * 65)
print(f"  Avg Queue Length : {np.mean([h['avg_queue_length'] for h in history]):.2f}")
print(f"  Avg Waiting Time : {np.mean([h['avg_waiting_time'] for h in history]):.2f}")
print(f"  Avg Throughput   : {np.mean([h['throughput'] for h in history]):.1f}")
print(f"  Avg Efficiency   : {np.mean([h['efficiency'] for h in history]):.4f}")
print("-" * 65)

## 10 · Manual Override Demo

In [ ]:
print("=" * 65)
print("  Manual Override Demo")
print("=" * 65)
print()

state = env.reset()
done = False
step_num = 0

while not done:
    step_num += 1

    # Inject manual override at step 200
    if step_num == 200:
        result = override_signal(env, lane=1, green_duration=50)
        print(f"\n  >>> STEP {step_num}: {result['message']}\n")

    # Cancel override at step 230
    if step_num == 230:
        result = cancel_override(env)
        print(f"  >>> STEP {step_num}: {result['message']}\n")

    action = agent.select_action(state, epsilon=0.0)
    next_state, reward, done, info = env.step(action)
    state = next_state

    if step_num % 50 == 0 or step_num == 1:
        q = info["queue_lengths"]
        ctrl = "OVERRIDE" if info["override_active"] else "AI"
        print(
            f"  Step {step_num:4d} | Queues: {q} | "
            f"Green: Lane {info['green_lane']} | "
            f"Control: {ctrl} | Departed: {info['total_departed']}"
        )

stats = env.get_episode_stats()
print()
print("-" * 65)
print("  Demo Summary")
print("-" * 65)
print(f"  Avg Queue Length : {stats['avg_queue_length']:.2f}")
print(f"  Avg Waiting Time : {stats['avg_waiting_time']:.2f}")
print(f"  Throughput       : {stats['throughput']}")
print(f"  Efficiency       : {stats['efficiency']:.4f}")
print("-" * 65)